In [ ]:
#-------------------------------------------------------RAYAN HASSAN & SOUMYAE TYAGI---------------------------------------------------#
#-----------------------------------------------------MINI PROJECT - DS 5110 SECTION 04------------------------------------------------#

!pip install kaggle

!kaggle datasets download cynthiarempel/amazon-us-customer-reviews-dataset

Dataset URL: https://www.kaggle.com/datasets/cynthiarempel/amazon-us-customer-reviews-dataset
License(s): other
amazon-us-customer-reviews-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip amazon-us-customer-reviews-dataset.zip 'amazon_reviews_us_Mobile_Electronics_v1_00.tsv' 'amazon_reviews_us_Major_Appliances_v1_00.tsv' 'amazon_reviews_us_Gift_Card_v1_00.tsv' 'amazon_reviews_us_Digital_Video_Games_v1_00.tsv' -d /content/dataset

Archive:  amazon-us-customer-reviews-dataset.zip
replace /content/dataset/amazon_reviews_us_Digital_Video_Games_v1_00.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset/amazon_reviews_us_Gift_Card_v1_00.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset/amazon_reviews_us_Major_Appliances_v1_00.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace /content/dataset/amazon_reviews_us_Mobile_Electronics_v1_00.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [ ]:
!pip install pyspark

In [ ]:
# We are only using a sample of the dataset due to storage limit in gcp
# We are using data from 4 .tsv files for good representation and merging them into 1 dataset
file_paths = [
    "/content/dataset/amazon_reviews_us_Digital_Video_Games_v1_00.tsv",
    "/content/dataset/amazon_reviews_us_Gift_Card_v1_00.tsv",
    "/content/dataset/amazon_reviews_us_Major_Appliances_v1_00.tsv",
    "/content/dataset/amazon_reviews_us_Mobile_Electronics_v1_00.tsv"
]

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("InClassWork9").getOrCreate()

df = spark.read.option("header", "true").option("delimiter", "\t").csv(file_paths)

for path in file_paths[1:]:
    df = df.union(spark.read.option("header", "true").option("delimiter", "\t").csv(path))

from pyspark.sql.functions import rand

df = df.withColumn("random", rand()).orderBy("random").drop("random")  # we did this to shuffle the rows so they're not ordered by categories

df.show(20)

+-----------+-----------+--------------+----------+--------------+--------------------+-------------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+
|marketplace|customer_id|     review_id|product_id|product_parent|       product_title|   product_category|star_rating|helpful_votes|total_votes|vine|verified_purchase|     review_headline|         review_body|review_date|
+-----------+-----------+--------------+----------+--------------+--------------------+-------------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+
|         US|   12545719| RN37XYRX6ISQO|B00D5R06HM|     725342975|Runshi Amazon Kin...| Mobile_Electronics|          1|            1|          2|   N|                Y|       DOES NOT WORK|I'm still trying ...| 2013-08-10|
|         US|   44963491|R1JPYVNAUV1UCP|B00DTWEOZ8|     869820667|Titanfall [Online...|Digital_Video_Games| 

In [ ]:
df.printSchema()

root
 |-- marketplace: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: string (nullable = true)
 |-- helpful_votes: string (nullable = true)
 |-- total_votes: string (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: string (nullable = true)



In [ ]:
# We are working with the review_body column to process and analyze reviews. Here is what it looks like before processing
df.select("review_body").show(20, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Data Preprocessing: Tokenization, cleaning, etc.

from pyspark.sql.functions import expr, explode, split, col, lower, regexp_replace
from pyspark.ml.feature import Tokenizer, StopWordsRemover


# Convert text to lowercase
df = df.withColumn("lowercase_review", lower(col("review_body")))

# Removing punctuation
df = df.withColumn("cleaned_sentence", regexp_replace(col("lowercase_review"), "[^a-zA-Z\s]", ""))

# Tokenize
df = df.withColumn("tokenized_sentence", split(col("cleaned_sentence"), "\s+"))


# Removing null values
df = df.filter(col("tokenized_sentence").isNotNull())


# Remove stop words
additional_stopwords = ["i"]

stopwords_remover = StopWordsRemover(inputCol="tokenized_sentence", outputCol="filtered_words")
stopwords_remover.setStopWords(stopwords_remover.getStopWords() + additional_stopwords)
df = stopwords_remover.transform(df)

df.select("cleaned_sentence","tokenized_sentence").show(20,truncate=False)



+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# this is what filtered_words column which is the final stage of the preprocessing:
df.select("filtered_words").show(20, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|filtered_words                                                                                                                                                                                                                                                                          

In [ ]:
# Filtering based on rating:

df_filtered = df.filter((col("star_rating") <= 2) | (col("star_rating") >= 4))
df_filtered.show(20)

+-----------+-----------+--------------+----------+--------------+--------------------+-------------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+
|marketplace|customer_id|     review_id|product_id|product_parent|       product_title|   product_category|star_rating|helpful_votes|total_votes|vine|verified_purchase|     review_headline|         review_body|review_date|    lowercase_review|    cleaned_sentence|  tokenized_sentence|      filtered_words|
+-----------+-----------+--------------+----------+--------------+--------------------+-------------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+--------------------+--------------------+--------------------+--------------------+
|         US|   12545719| RN37XYRX6ISQO|B00D5R06HM|     725342975|Runshi Amazon

In [ ]:
!pip install textblob

In [ ]:
# Sentiment Analysis
# We are using TextBlob for a simpler NLP because actual Spark NLP won't work after several trials
# Of course, the accuracy is not going to be 100%, so some sentiments might not be correct

from textblob import TextBlob
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def get_sentiment(text):
    text = " ".join(text)
    analysis = TextBlob(text)
    polarity = analysis.sentiment.polarity
    if polarity > 0:
        sentiment = "positive"
    elif polarity < 0:
        sentiment = "negative"
    else:
        sentiment = "neutral"

    return sentiment

sentiment_udf = udf(get_sentiment, StringType())


df_filtered = df_filtered.withColumn("sentiment", sentiment_udf(col("filtered_words")))



df_filtered.select("star_rating", "sentiment").show(15,truncate=False)


+-----------+---------+
|star_rating|sentiment|
+-----------+---------+
|1          |negative |
|4          |positive |
|5          |positive |
|5          |positive |
|5          |positive |
|5          |positive |
|5          |positive |
|5          |positive |
|5          |neutral  |
|5          |positive |
|1          |negative |
|1          |neutral  |
|5          |positive |
|5          |positive |
|5          |positive |
+-----------+---------+
only showing top 15 rows



In [ ]:
# Key phrase extraction

# Here we are using SpaCy library for key word extraction!

import spacy
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, StringType


nlp = spacy.load("en_core_web_sm")


def extract_noun_phrases_spacy(words):
    if not words:
        return []
    text = " ".join(words)
    doc = nlp(text)
    return [chunk.text for chunk in doc.noun_chunks]


noun_phrases_udf = udf(extract_noun_phrases_spacy, ArrayType(StringType()))


df_filtered = df_filtered.withColumn("key_phrases", noun_phrases_udf(col("filtered_words")))


df_filtered.select("sentiment", "key_phrases").show(20, truncate=False)


/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|sentiment|key_phrases                                                                                                                                                                                                                                                                                                      |
+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|negative |[i, figure amazon, one guys, small 

In [ ]:
# Because the next part (summarization) will require word count, we decided to work on a smaller sample dataset (due to time compute limitation)
# we tried doing word count on the original dataset but it takes A LOT, and we mean A LOT of time (we waited for 2 hours and it still hadn't finished running).
# we really hope you can understand and we hope it won't affect our grade!


# Summarize positive reviews
from pyspark.sql.functions import explode, col

sample_df = df_filtered.limit(500)


positive_summary = (sample_df.filter(col("sentiment") == "positive").withColumn("key_phrase", explode(col("key_phrases"))).groupBy("key_phrase")
  .count().orderBy(col("count").desc()))

positive_summary = positive_summary.withColumnRenamed("key_phrase", "key_phrases")

positive_summary.show(10, truncate=False)


+-----------+-----+
|key_phrases|count|
+-----------+-----+
|i          |42   |
|something  |17   |
|gift       |13   |
|amazon     |13   |
|gift card  |12   |
|someone    |12   |
|that       |12   |
|us         |12   |
|anything   |11   |
|clothes    |10   |
+-----------+-----+
only showing top 10 rows



In [ ]:
# Summarize negative reviews
negative_summary = (sample_df.filter(col("sentiment") == "negative").withColumn("key_phrase", explode(col("key_phrases"))).groupBy("key_phrase")
    .count().orderBy(col("count").desc()))

negative_summary = negative_summary.withColumnRenamed("key_phrase", "key_phrases")

negative_summary.show(10, truncate=False)

+-----------+-----+
|key_phrases|count|
+-----------+-----+
|i          |15   |
|game       |9    |
|something  |6    |
|product    |4    |
|amazon     |3    |
|you        |3    |
|case       |3    |
|help       |2    |
|everyone   |2    |
|anything   |2    |
+-----------+-----+
only showing top 10 rows



In [ ]:
# Summarized feedback

from pyspark.sql.functions import explode, collect_list, concat_ws, col

sentiment_key_phrases = (
    sample_df.withColumn("key_phrase", explode(col("key_phrases")))
    .groupBy("sentiment", "key_phrase")
    .count()
    .orderBy("sentiment", col("count").desc())
)

summarized_feedback = (
    sentiment_key_phrases.groupBy("sentiment")
    .agg(collect_list("key_phrase").alias("key_phrases_list"))
    .withColumn("summarized_feedback", concat_ws(", ", col("key_phrases_list")))
    .select("sentiment", "summarized_feedback")
)

summarized_feedback.show(20, truncate=False)

+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# Aggregation and Insights

# First we show how many negative, neutral, and positive reviews there are in each category respectively
category_sentiment_counts = (
    sample_df.groupBy("product_category", "sentiment")
    .count()
    .withColumnRenamed("count", "review_count")
)

category_sentiment_counts = category_sentiment_counts.orderBy("product_category", "sentiment")
category_sentiment_counts.show(20, truncate=False)

+-------------------+---------+------------+
|product_category   |sentiment|review_count|
+-------------------+---------+------------+
|Digital_Video_Games|negative |24          |
|Digital_Video_Games|neutral  |6           |
|Digital_Video_Games|positive |50          |
|Gift Card          |negative |6           |
|Gift Card          |neutral  |29          |
|Gift Card          |positive |162         |
|Major Appliances   |negative |23          |
|Major Appliances   |neutral  |8           |
|Major Appliances   |positive |83          |
|Mobile_Electronics |negative |14          |
|Mobile_Electronics |neutral  |9           |
|Mobile_Electronics |positive |86          |
+-------------------+---------+------------+



In [ ]:
# Now we calculate total reviews per category in the sample df

category_totals = (
    sample_df.groupBy("product_category")
    .count()
    .withColumnRenamed("count", "total_reviews")
)

category_totals.show(20, truncate=False)


+-------------------+-------------+
|product_category   |total_reviews|
+-------------------+-------------+
|Mobile_Electronics |109          |
|Digital_Video_Games|80           |
|Gift Card          |197          |
|Major Appliances   |114          |
+-------------------+-------------+



In [ ]:
# Here we join category_sentiment_counts with category_totals to calculate percentages
from pyspark.sql.functions import round

category_sentiment_percentage = (
    category_sentiment_counts.join(category_totals, on="product_category")
    .withColumn("percentage", round((col("review_count") / col("total_reviews")) * 100, 2))
    .drop("total_reviews")
)

category_sentiment_percentage.orderBy("product_category", "sentiment").show(20, truncate=False)


+-------------------+---------+------------+----------+
|product_category   |sentiment|review_count|percentage|
+-------------------+---------+------------+----------+
|Digital_Video_Games|negative |24          |30.0      |
|Digital_Video_Games|neutral  |6           |7.5       |
|Digital_Video_Games|positive |50          |62.5      |
|Gift Card          |negative |6           |3.05      |
|Gift Card          |neutral  |29          |14.72     |
|Gift Card          |positive |162         |82.23     |
|Major Appliances   |negative |23          |20.18     |
|Major Appliances   |neutral  |8           |7.02      |
|Major Appliances   |positive |83          |72.81     |
|Mobile_Electronics |negative |14          |12.84     |
|Mobile_Electronics |neutral  |9           |8.26      |
|Mobile_Electronics |positive |86          |78.9      |
+-------------------+---------+------------+----------+



In [ ]:
# Finding common issues/key phrases or words

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, explode

category_key_phrases = (
    sample_df.withColumn("key_phrase", explode(col("key_phrases")))
    .groupBy("product_category", "sentiment", "key_phrase")
    .count()
    .orderBy("product_category", "sentiment", col("count").desc())
)

window_spec = Window.partitionBy("product_category", "sentiment").orderBy(col("count").desc())


top_key_phrases = (
    category_key_phrases
    .withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") == 1)
    .drop("rank")
)

top_key_phrases.show(20, truncate=False)


+-------------------+---------+----------+-----+
|product_category   |sentiment|key_phrase|count|
+-------------------+---------+----------+-----+
|Digital_Video_Games|negative |game      |9    |
|Digital_Video_Games|neutral  |charm     |2    |
|Digital_Video_Games|positive |game      |5    |
|Gift Card          |negative |wrong cash|1    |
|Gift Card          |neutral  |excelente |5    |
|Gift Card          |positive |gift      |13   |
|Major Appliances   |negative |i         |5    |
|Major Appliances   |neutral  |ice maker |1    |
|Major Appliances   |positive |i         |19   |
|Mobile_Electronics |negative |case      |3    |
|Mobile_Electronics |neutral  |product   |2    |
|Mobile_Electronics |positive |i         |9    |
+-------------------+---------+----------+-----+



Overview

This report analyzes customer feedback trends using the provided dataset of Amazon US customer reviews. The data focuses on specific product categories, exploring customer sentiments, feedback volume, and ratings to derive actionable insights for business improvements.

Key Insights

1.⁠ ⁠Sentiment Analysis

Positive Feedback: A significant portion of reviews have high ratings (4 or 5 stars), indicating general customer satisfaction. The summarized feedback table suggests that positive sentiments are associated with good features in products such as price, accessibility, simple usability, etc.

Negative Feedback: Low ratings (1 or 2 stars) are primarily associated with issues like defective products, delayed deliveries, and poor customer service.

Neutral Feedback: Three-star reviews often indicate ambiguity in customer satisfaction, suggesting areas for improvement.



Actionable Recommendations

To improve customer satisfaction and address recurring issues, focus on enhancing quality control to eliminate product defects and design flaws, such as faulty components and unclear documentation. Simplify user manuals with visual aids and step-by-step instructions to improve usability. Strengthen customer support with faster issue resolution and proactive assistance for compatibility problems and returns. Increase transparency in product descriptions to manage customer expectations and avoid misleading claims. Use feedback-driven redesigns to enhance product reliability and functionality. Prepare for seasonal demand spikes with optimized resources and inventory to ensure timely support and fulfillment. These actions will improve customer experience, brand loyalty, and operational efficiency.